In [1]:
import pathlib
import shutil

import numpy as np
from image_analysis_3D.file_utils.notebook_init_utils import init_notebook
from manim import *

git_root, in_notebook = init_notebook()

In [2]:
# ---------------------------------------------------------------------------
# Terrain: inverted "distance map" of two touching blobs
# ---------------------------------------------------------------------------
C1 = np.array([-1.5, 0.0])  # blob 1 centre  -> basin 1 (TEAL)
C2 = np.array([1.5, 0.0])  # blob 2 centre  -> basin 2 (MAROON)

U_RANGE = (-3.5, 3.5)
V_RANGE = (-2.5, 2.5)
RES_U, RES_V = 42, 30  # even counts -> a grid line sits exactly on u = 0


def height_func(u, v):
    d1 = (u - C1[0]) ** 2 + (v - C1[1]) ** 2
    d2 = (u - C2[0]) ** 2 + (v - C2[1]) ** 2
    return 2.5 - 2.0 * (np.exp(-d1 / 2.0) + np.exp(-d2 / 2.0))


MIN_Z = height_func(*C1)  # ~0.48  regional minima
SADDLE_Z = height_func(0, 0)  # ~1.20  lowest point of the ridge (where basins meet)
TOP_Z = 2.1  # final flood level

# Precomputed grid for the water mesh
US = np.linspace(*U_RANGE, RES_U + 1)
VS = np.linspace(*V_RANGE, RES_V + 1)
H = height_func(US[:, None], VS[None, :])


def clip_cell(quad, level):
    """Clip a grid cell to the flooded region h(u, v) < level, interpolating the
    shoreline along cell edges (Sutherland-Hodgman).

    Parameters
    ----------
    quad : list of (float, float)
        Corners of the grid cell, as (u, v) pairs.
    level : float
        Flood height to clip against.

    Returns
    -------
    list of (float, float)
        Vertices of the clipped polygon, in order.
    """
    out = []
    for i in range(len(quad)):
        p, q = quad[i], quad[(i + 1) % len(quad)]
        fp, fq = level - height_func(*p), level - height_func(*q)
        if fp >= 0:
            out.append(p)
        if (fp >= 0) != (fq >= 0):
            out.append(p + fp / (fp - fq) * (q - p))
    return out


def make_water(level, side, color, opacity=0.65):
    """Water surface of one basin: only where the terrain is below `level`,
    and only on that basin's side of the watershed (u < 0 or u > 0).

    Parameters
    ----------
    level : float
        Flood height.
    side : int
        Which basin to draw water for: negative for u < 0, positive for u > 0.
    color : str
        Fill color of the water surface.
    opacity : float, optional
        Fill opacity of the water surface, by default 0.65.

    Returns
    -------
    VGroup
        Polygons making up the water surface.
    """
    water = VGroup()
    for i in range(RES_U):
        if side < 0 and US[i] >= 0:
            continue
        if side > 0 and US[i + 1] <= 0:
            continue
        for j in range(RES_V):
            if H[i : i + 2, j : j + 2].min() >= level:
                continue
            quad = [
                np.array(p)
                for p in (
                    (US[i], VS[j]),
                    (US[i + 1], VS[j]),
                    (US[i + 1], VS[j + 1]),
                    (US[i], VS[j + 1]),
                )
            ]
            poly = clip_cell(quad, level)
            if len(poly) < 3:
                continue
            water.add(
                Polygon(
                    *[[p[0], p[1], level + 0.01] for p in poly],
                    fill_color=color,
                    fill_opacity=opacity,
                    stroke_width=0,
                )
            )
    return water


def make_dam(level, n=60):
    """Dam wall on the ridge (u = 0): from the terrain up to the water level,
    only where the water would otherwise spill from one basin into the other.

    Parameters
    ----------
    level : float
        Flood height.
    n : int, optional
        Number of segments to divide the ridge into, by default 60.

    Returns
    -------
    VGroup
        Polygons making up the dam wall.
    """
    dam = VGroup()
    if level <= SADDLE_Z:
        return dam
    top = level + 0.04
    vs = np.linspace(*V_RANGE, n + 1)
    for a, b in zip(vs[:-1], vs[1:]):
        ha, hb = height_func(0, a), height_func(0, b)
        if min(ha, hb) >= level:
            continue
        dam.add(
            Polygon(
                [0, a, min(ha, top)],
                [0, b, min(hb, top)],
                [0, b, top],
                [0, a, top],
                fill_color=WHITE,
                fill_opacity=0.9,
                stroke_width=0,
            )
        )
    return dam


class Watershed2D(ThreeDScene):
    def show_caption(self, text, old=None, corner=UL, **kwargs):
        cap = Text(text, font_size=22, color=WHITE, **kwargs).to_corner(corner)
        self.add_fixed_in_frame_mobjects(cap)
        self.remove(cap)
        anims = [FadeIn(cap, shift=0.2 * DOWN)]
        if old is not None:
            anims.append(FadeOut(old))
        self.play(*anims, run_time=1.2)
        return cap

    def construct(self):
        self.set_camera_orientation(phi=0 * DEGREES, theta=-90 * DEGREES)
        self.camera.background_color = "#0B0F19"

        # 1. Terrain ----------------------------------------------------------
        surface = Surface(
            lambda u, v: np.array([u, v, height_func(u, v)]),
            u_range=list(U_RANGE),
            v_range=list(V_RANGE),
            resolution=(RES_U, RES_V),
            fill_opacity=0.85,
            checkerboard_colors=["#1E293B", "#0F172A"],
            stroke_color="#38BDF8",
            stroke_width=0.4,
        )
        self.add(surface)
        cap = self.show_caption("Distance map of two touching blobs")
        self.wait(1.5)
        self.move_camera(phi=65 * DEGREES, theta=-55 * DEGREES, run_time=3)
        cap = self.show_caption("Inverted: blob centres become valleys", cap)
        self.wait(1.2)

        # 2. Markers at the regional minima ----------------------------------
        min1 = Dot3D(point=[*C1, MIN_Z + 0.1], color=TEAL, radius=0.12)
        min2 = Dot3D(point=[*C2, MIN_Z + 0.1], color=MAROON, radius=0.12)
        self.play(FadeIn(min1, scale=0.5), FadeIn(min2, scale=0.5))
        cap = self.show_caption("Markers placed at the regional minima", cap)
        self.wait(1.5)

        # 3. Flood each basin from its own marker -----------------------------
        level = ValueTracker(MIN_Z + 0.02)
        water1 = always_redraw(lambda: make_water(level.get_value(), -1, TEAL))
        water2 = always_redraw(lambda: make_water(level.get_value(), +1, MAROON))
        dam = always_redraw(lambda: make_dam(level.get_value()))
        self.add(water1, water2, dam)

        cap = self.show_caption("Flooding: water rises from each marker", cap)
        self.play(
            level.animate.set_value(SADDLE_Z - 0.01), run_time=4, rate_func=linear
        )

        # 4. Basins meet at the saddle point ---------------------------------
        saddle = Dot3D(point=[0, 0, SADDLE_Z + 0.05], color=YELLOW, radius=0.1)
        cap = self.show_caption("The two basins are about to merge at the saddle", cap)
        self.play(FadeIn(saddle, scale=0.3))
        self.play(Flash(saddle, color=YELLOW, flash_radius=0.4))
        self.wait(1.5)

        # 5. Keep flooding; a dam grows wherever the water would spill over ---
        cap = self.show_caption("Build a dam where they meet: the watershed line", cap)
        self.play(FadeOut(saddle), run_time=0.4)
        self.play(level.animate.set_value(TOP_Z), run_time=5, rate_func=linear)
        self.wait(1.5)

        for m in (water1, water2, dam):
            m.clear_updaters()

        # Crest of the dam / ridge = watershed line
        ws_line = ParametricFunction(
            lambda t: np.array([0, t, max(height_func(0, t), TOP_Z) + 0.06]),
            t_range=[*V_RANGE],
            color=WHITE,
            stroke_width=5,
        )
        self.play(Create(ws_line), run_time=1.5)

        # 6. Top-down: the result is a segmentation ---------------------------
        segmented = surface.copy()
        for face in segmented:
            x = face.get_center()[0]
            face.set_fill(TEAL_D if x < 0 else MAROON_D, opacity=0.85)
            face.set_stroke(width=0)

        self.move_camera(phi=0 * DEGREES, theta=-90 * DEGREES, run_time=3)
        self.play(
            FadeOut(water1),
            FadeOut(water2),
            FadeOut(dam),
            FadeOut(min1),
            FadeOut(min2),
            Transform(surface, segmented),
            run_time=2,
        )
        self.play(ws_line.animate.set_z(height_func(0, 0) + 1.5))  # keep it on top
        cap = self.show_caption("Result: two separated objects", cap)

        lbl1 = Text("Region 1", font_size=26, color=WHITE)
        lbl2 = Text("Region 2", font_size=26, color=WHITE)
        self.add_fixed_in_frame_mobjects(lbl1, lbl2)
        lbl1.move_to(LEFT * 3 + DOWN * 3)
        lbl2.move_to(RIGHT * 3 + DOWN * 3)
        self.remove(lbl1, lbl2)
        self.play(FadeIn(lbl1), FadeIn(lbl2))
        self.wait(3)

In [3]:
# call Watershed2D scene
scene = Watershed2D()
scene.render()

[11/06/26 08:02:58] INFO     Animation 0 : Partial movie file written in                   ]8;id=13804805;file:///home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pipeline/.venv/lib/python3.13/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=13804806;file:///home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pipeline/.venv/lib/python3.13/site-packages/manim/scene/scene_file_writer.py#173\173]8;;\
                             '/home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pip                         
                             eline/2.segment_images/notebooks/media/videos/1080p60/partial                         
                             _movie_files/Watershed2D/91e63396e52020dbbb635580ecfe464b2c57                         
                             6694a9818df5c2e7e7a3e27af254.mp4'                                                     

[11/06/26 08:03:02] INFO     Animation 1 : Partial movie file written in                   ]8;id=13804811;file:///home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pipeline/.venv/lib/python3.13/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=13804812;file:///home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pipeline/.venv/lib/python3.13/site-packages/manim/scene/scene_file_writer.py#173\173]8;;\
                             '/home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pip                         
                             eline/2.segment_images/notebooks/media/videos/1080p60/partial                         
                             _movie_files/Watershed2D/61202f229b42c99c0be97ef035f982b3a726                         
                             226a6815e9db4ca4e3d880cb81fb.mp4'                                                     

[11/06/26 08:05:21] INFO     Animation 2 : Partial movie file written in                   ]8;id=13804817;file:///home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pipeline/.venv/lib/python3.13/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=13804818;file:///home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pipeline/.venv/lib/python3.13/site-packages/manim/scene/scene_file_writer.py#173\173]8;;\
                             '/home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pip                         
                             eline/2.segment_images/notebooks/media/videos/1080p60/partial                         
                             _movie_files/Watershed2D/96705455df39651ee61d2e8323fe0f1b1fa1                         
                             f0aaf6f335126bc1b66ee1e81e22.mp4'                                                     

[11/06/26 08:06:08] INFO     Animation 3 : Partial movie file written in                   ]8;id=13804823;file:///home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pipeline/.venv/lib/python3.13/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=13804824;file:///home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pipeline/.venv/lib/python3.13/site-packages/manim/scene/scene_file_writer.py#173\173]8;;\
                             '/home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pip                         
                             eline/2.segment_images/notebooks/media/videos/1080p60/partial                         
                             _movie_files/Watershed2D/df02094bc0cee4baa727f16d5bf1bda9bbc8                         
                             0aaa218c25512774fb7bb11e5b27.mp4'                                                     

[11/06/26 08:06:10] INFO     Animation 4 : Partial movie file written in                   ]8;id=13804829;file:///home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pipeline/.venv/lib/python3.13/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=13804830;file:///home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pipeline/.venv/lib/python3.13/site-packages/manim/scene/scene_file_writer.py#173\173]8;;\
                             '/home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pip                         
                             eline/2.segment_images/notebooks/media/videos/1080p60/partial                         
                             _movie_files/Watershed2D/95d2c1e4da74d0fc132d95542d3b6fd383ce                         
                             af9de504c58542690ff741b9716a.mp4'                                                     

[11/06/26 08:06:16] INFO     Animation 5 : Partial movie file written in                   ]8;id=13804835;file:///home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pipeline/.venv/lib/python3.13/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=13804836;file:///home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pipeline/.venv/lib/python3.13/site-packages/manim/scene/scene_file_writer.py#173\173]8;;\
                             '/home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pip                         
                             eline/2.segment_images/notebooks/media/videos/1080p60/partial                         
                             _movie_files/Watershed2D/b4afa09161bdfe47353ac97a0444ce2d6312                         
                             576392405467688f3db20575d06d.mp4'                                                     

... (19 outputs left out)

                    INFO     Combining to Movie file.                                      ]8;id=13804956;file:///home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pipeline/.venv/lib/python3.13/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=13804957;file:///home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pipeline/.venv/lib/python3.13/site-packages/manim/scene/scene_file_writer.py#877\877]8;;\

                    INFO                                                                   ]8;id=13804963;file:///home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pipeline/.venv/lib/python3.13/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=13804964;file:///home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pipeline/.venv/lib/python3.13/site-packages/manim/scene/scene_file_writer.py#999\999]8;;\
                             File ready at                                                                         
                             '/home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pip                         
                             eline/2.segment_images/notebooks/media/videos/1080p60/Watersh                         
                             ed2D.mp4'                                                                             
                                                                                                                   

                    INFO     Rendered Watershed2D                                                    ]8;id=13804971;file:///home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pipeline/.venv/lib/python3.13/site-packages/manim/manager.py\manager.py]8;;\:]8;id=13804972;file:///home/lippincm/Documents/fork2_NF1_3D_organoid_profiling_pipeline/.venv/lib/python3.13/site-packages/manim/manager.py#226\226]8;;\
                             Played 25 animations                                                                  

False

In [4]:
dirs_to_remove = [
    pathlib.Path(
        f"{git_root}/2.segment_images/notebooks/media/videos/1080p60/partial_movie_files"
    ).resolve(),
    pathlib.Path(f"{git_root}/2.segment_images/notebooks/media/texts/").resolve(),
]
for dir_path in dirs_to_remove:
    if dir_path.exists():
        shutil.rmtree(dir_path)